<a href="https://colab.research.google.com/github/Gedion-Sang/Cyber_Shujaa/blob/main/Gedion_Sang_MLOPs_CS_DA03_26010.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

***Title: Classification Models Assignment***

***Name: Gedion Sang***

***ID: CS-DA03-26010***

***Date: 13/03/2026***

# **MLOPs**

## Introduction


This notebook demonstrates an end-to-end Machine Learning workflow, including data loading, preprocessing, model training, hyperparameter tuning using GridSearchCV, evaluation, and saving the final model. We'll be using the California Housing dataset.

### 1.	Load the dataset

We start by importing all necessary libraries for data manipulation, machine learning, and model evaluation.

In [1]:
# 📦 Import libraries
import pandas as pd
import pickle
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error

The California Housing dataset is loaded using `fetch_california_housing` from `sklearn.datasets`. The features (X) and target (y) are returned as pandas DataFrames/Series.

In [2]:
X, y = fetch_california_housing(return_X_y=True, as_frame=True)
print(X.shape, y.shape)


(20640, 8) (20640,)


### 2. Train-test split

The dataset is split into training and testing sets to evaluate the model's performance on unseen data. A `test_size` of 20% and `random_state` of 42 are used for reproducibility.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(16512, 8) (4128, 8) (16512,) (4128,)


### 3. Preprocessing: Imputation + Scaling for numerical features

A `numeric_transformer` pipeline is defined for numerical features. It includes `SimpleImputer` to handle missing values (using the mean strategy) and `StandardScaler` to normalize features.

In [4]:
numeric_features = X.columns  # all are numerical
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

###  4. Combine preprocessing using ColumnTransformer

A `ColumnTransformer` is used to apply the `numeric_transformer` to all numerical features identified (which are all columns in this dataset).

In [5]:
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features)
])

### 5. Build pipeline: preprocessing + KNN

A final `Pipeline` is constructed, combining the `preprocessor` (for imputation and scaling) and the `KNeighborsRegressor` model.

In [6]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('knn', KNeighborsRegressor())
])

### 6. Define hyperparameter grid

A dictionary `param_grid` is defined to specify the hyperparameters to search over for the `KNeighborsRegressor` model. This includes the number of neighbors (`n_neighbors`), weight function (`weights`), and power parameter for the Minkowski metric (`p`).

In [7]:
param_grid = {
    'knn__n_neighbors': [3, 5, 7, 9],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2]
}
print(param_grid)

{'knn__n_neighbors': [3, 5, 7, 9], 'knn__weights': ['uniform', 'distance'], 'knn__p': [1, 2]}


### 7. Apply GridSearchCV with 5-fold cross-validation

A `GridSearchCV` object is initialized with the pipeline, hyperparameter grid, 5-fold cross-validation, and `r2` as the scoring metric. `n_jobs=-1` uses all available processors for faster computation.

In [8]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    verbose=1,
    n_jobs=-1
)

### 8. Fit the model

The `GridSearchCV` is fit to the training data (`X_train`, `y_train`) to find the best combination of hyperparameters.

In [9]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer()),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         Index(['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup',
       'Latitude', 'Longitude'],
      dtype='object'))])),
                                       ('knn', KNeighborsRegressor())]),
             n_jobs=-1,
             param_grid={'knn__n_neighbors': [3, 5, 7, 9], 'knn__p': [1, 2],
                         'knn__weights': ['uniform', 'distance']},
             scoring='r2', verbose=1)

### 9. Evaluate on test set


The best model found by `GridSearchCV` is used to make predictions on the test set (`X_test`). Then, R² score, Mean Squared Error (MSE), and Root Mean Squared Error (RMSE) are calculated to evaluate the model's performance.

In [10]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5

### 10. Print results

The best hyperparameters, best cross-validation R² score, and the test set R², MSE, and RMSE are printed to the console.

In [11]:
print("Best Parameters:", grid_search.best_params_)
print("Best CV R² Score:", grid_search.best_score_)
print("Test R² Score:", r2)
print("Test MSE:", mse)
print("Test RMSE:", rmse)

Best Parameters: {'knn__n_neighbors': 9, 'knn__p': 1, 'knn__weights': 'distance'}
Best CV R² Score: 0.731266870986164
Test R² Score: 0.72210916268423
Test MSE: 0.3641506481894662
Test RMSE: 0.6034489607162036


### 11. Save the pipeline

The trained `best_model` (which is the entire pipeline with optimal hyperparameters) is saved to a file named 'california_knn_pipeline.pkl' using Python's `pickle` module. This allows for later deployment or use of the model without retraining.

In [12]:
with open('california_knn_pipeline.pkl', 'wb') as f:pickle.dump(best_model, f)
print("📦 Final pipeline saved to 'california_knn_pipeline.pkl'")

📦 Final pipeline saved to 'california_knn_pipeline.pkl'
